# Where do schizophrenia-associated CNVs converge?

Run top to bottom in a fresh project kernel. `PCNET2.2` is the article
network; set `CNV_NETWORK=STRING_high_conf` for the supporting interactome check.

```bash
.venv/bin/python scripts/run_network_notebooks.py --networks PCNET2.2 --notebooks cnv_convergence
```

Results go to `results/<network>/`; each executed notebook and log has a unique
attempt directory under `results/executions/`. The minimum retained seed count
is set in the next cell. See the [notebook guide](README.md) and
[methods](../docs/methods.md).


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cnv_overlap.pipeline import AnalysisConfig, run
from cnv_overlap.visualization import pair_heatmap


ROOT = Path.cwd()
if ROOT.name == "notebooks":  # opened from notebooks/ in Jupyter
    ROOT = ROOT.parent
if not (ROOT / "configs").is_dir():
    raise RuntimeError("Start this notebook with the project root as working directory")
CONFIG_PATH = Path(os.environ.get("CNV_CONFIG", ROOT / "configs/cnv_convergence.json"))
import sys
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from network_inputs import load_network_inputs
from results_layout import results_dir
# Use STRING_high_conf for the supporting interactome check.
NETWORK = os.environ.get("CNV_NETWORK", None if "CNV_CONFIG" in os.environ else "PCNET2.2")
# Minimum retained seeds per gene set.
MIN_SEEDS = 3
spec, root, inputs, provenance = load_network_inputs(CONFIG_PATH, NETWORK, min_seeds=MIN_SEEDS)
RESULTS_ROOT = root / spec["results_root"]
print("Network:", NETWORK or "custom snapshot", "Results:", RESULTS_ROOT, "Cache:", root / spec["cache_dir"])
config = AnalysisConfig.from_dict(spec["analysis"])
if config.comparisons != "cnv_cnv":
    raise ValueError("This notebook requires analysis.comparisons=cnv_cnv")
A, nodes, symbols, entities, coordinates, regions = inputs
print(f"{len(nodes):,} nodes, {A.nnz // 2:,} edges; {len(entities)} gene sets")
print("Supporting pairs: overlap size; NetColoc normal-tail p-values; overlap policy:", config.overlap)
print("Gene-label permutations:", config.permutation_reps)

from notebook_analysis import SCZ_TARGETS, run_convergence


In [ ]:
RUN = run(*inputs, config=config, output_root=root/spec["output_root"],
          cache_dir=root/spec["cache_dir"], provenance=provenance)
manifest = json.loads((RUN / "manifest.json").read_text())
assert manifest["status"] == "complete"
print(RUN)
print((RUN / "summary.md").read_text())
from cnv_overlap.results import notebook_execution_dir
FIGURES = notebook_execution_dir(RESULTS_ROOT, NETWORK) / "reports" / "cnv_convergence"
FIGURES.mkdir(parents=True, exist_ok=True)


## Input QC and tested hypotheses

Inspect seed losses, coordinate coverage and the hypothesis list. Coordinates document CNV content; all retained network genes are eligible for degree-matched references.
Only CNV loci contribute to the tested family.


In [ ]:
from cnv_overlap.results import resolve_path
qc = pd.read_csv(resolve_path(RUN / "seed_qc.tsv"), sep="\t")
display(qc)
display(pd.read_csv(resolve_path(RUN / "seed_overlap.tsv"), sep="\t").set_index("entity"))
display(pd.read_csv(resolve_path(RUN / "hypotheses.tsv"), sep="\t").groupby("family").size().rename("tests"))


## Multi-CNV convergence

Count CNVs with NPS above the individual threshold at each gene. The global
statistic sums `k*(k-1)/2` over genes. Joint randomizations preserve each CNV's
seed count in every degree bin; checkpoint blocks support resumption.

Gene `p_empirical` values are unadjusted; `p_maxT` uses the maximum count per
draw and is calibrated under the complete joint seed null. SCZ endpoints sum
convergence weights within fixed SCHEMA/GWAS footprints, with BH correction
across the two endpoints. See [methods](../docs/methods.md#multi-cnv-convergence).


In [ ]:
from cnv_overlap.results import resolve_path
# Load SCZ targets using the validated snapshot, with the same tested network.
scz_entities = {k: v for k, v in entities.items() if k in SCZ_TARGETS and v["kind"] == "target"}
if "CNV_CONFIG" not in os.environ:
    from cnv_overlap.provenance import graph_hash
    scz_spec, scz_root, scz_inputs, scz_provenance = load_network_inputs(ROOT / "configs/cnv_disorder_overlap.json", NETWORK, min_seeds=MIN_SEEDS)
    if graph_hash(scz_inputs[0], scz_inputs[1]) != graph_hash(A, nodes):
        raise ValueError("SCZ targets require the same network universe as CNV convergence")
    scz_entities = {k: v for k, v in scz_inputs[3].items() if k in SCZ_TARGETS and v["kind"] == "target"}
if not scz_entities:
    print("No schema/gwaspri targets in these custom inputs: SCZ relevance endpoints are unavailable.")
JOINT_REPS = config.permutation_reps
CONVERGENCE = run_convergence(inputs, config=config, cache_dir=root/spec["cache_dir"],
    output_root=results_dir(RESULTS_ROOT, "multi_cnv_convergence")/"degree", reps=JOINT_REPS, scz_entities=scz_entities)
convergence_genes = pd.read_csv(resolve_path(CONVERGENCE / "gene_convergence.tsv"), sep="\t")
convergence_tests = pd.read_csv(resolve_path(CONVERGENCE / "convergence_tests.tsv"), sep="\t")
display(convergence_tests)
display(convergence_genes.sort_values(["n_cnvs", "p_empirical"], ascending=[False, True]).head(50))
print("Full results:", CONVERGENCE)
fig, ax = plt.subplots(figsize=(7, 4))
convergence_genes.n_cnvs.value_counts().sort_index().plot.bar(ax=ax)
ax.set(xlabel="Number of CNVs reaching a gene", ylabel="Network genes", title="Multi-CNV recurrence: all network genes")
fig.tight_layout()
fig.savefig(resolve_path(FIGURES / "multi_cnv_recurrence.png"), dpi=200)
plt.show()


## Archived shared-core analysis (off by default)

The main pipeline now runs `3_scz_landscape` then `4_cnv_reactome_profiles`.
These historical sections run only when `CNV_RUN_LEGACY_SHARED_CORE=1`.


In [ ]:
if os.environ.get("CNV_RUN_LEGACY_SHARED_CORE") == "1":
    from cnv_overlap.results import resolve_path
    from shared_network_pipeline import run_shared_network_pipeline
    from shared_network_enrichment import LIBRARIES
    MIN_SHARED_CNVS = 3
    network_entities = entities | scz_entities
    SCZ_ENRICHMENT = None
    SCZ_SENSITIVITY = None
    if any(n in network_entities and network_entities[n]["kind"] == "target" for n in SCZ_TARGETS):
        annotation_dir = root/"input/annotations"
        missing_libraries = [n for n in LIBRARIES.values() if not (annotation_dir/n).exists()]
        both_targets = all(n in network_entities and network_entities[n]["kind"] == "target" for n in SCZ_TARGETS)
        do_enrichment = both_targets and not missing_libraries
        if missing_libraries:
            print("Shared-core enrichment unavailable; missing local libraries:", missing_libraries)
        workflow = run_shared_network_pipeline(
            (A, nodes, symbols, network_entities, coordinates, regions), config=config,
            cache_dir=root/spec["cache_dir"], output_root=RESULTS_ROOT,
            annotation_dir=annotation_dir, min_cnvs=MIN_SHARED_CNVS, enrich=do_enrichment, collapse=False, )
        SCZ_NETWORKS, SCZ_ENRICHMENT = workflow["networks"], workflow["enrichment"]
        SCZ_SENSITIVITY = workflow["sensitivity"]
        network_index = pd.read_csv(resolve_path(SCZ_NETWORKS/"network_index.tsv"), sep="\t")
        display(network_index)
        shared_genes = pd.read_csv(resolve_path(SCZ_NETWORKS/"gene_membership.tsv"), sep="\t")
        if "shared_core" in shared_genes:
            display(shared_genes.loc[shared_genes.shared_core].sort_values("n_cnvs_both", ascending=False))
            from IPython.display import Image
            core = network_index[network_index.kind == "shared_core"].iloc[0]
            display(Image(filename=str(resolve_path(SCZ_NETWORKS/core.directory/"network.png"))))
        else:
            print("Joint core unavailable: both SCHEMA and GWAS-prioritized targets are required.")
        print("Complete network exports:", SCZ_NETWORKS)
    else:
        print("SCZ networks unavailable: no SCHEMA/GWAS-prioritized target sets supplied.")


### Leiden clusters and annotations

Partition the shared core with Leiden (seed 42, resolution 1). GO/Reactome ORA
uses the eligible network background and BH correction across cluster–term
tests. Singletons are listed in `excluded_singletons.tsv`. Annotations are
descriptive after network selection.


In [ ]:
if os.environ.get("CNV_RUN_LEGACY_SHARED_CORE") == "1":
    from cnv_overlap.results import resolve_path
    SCZ_LEIDEN = None
    if "SCZ_NETWORKS" in globals() and both_targets:
        from shared_network_leiden import annotate_leiden
        SCZ_LEIDEN = annotate_leiden(SCZ_NETWORKS, annotation_dir=annotation_dir,
            output_root=results_dir(RESULTS_ROOT, "scz_network_leiden"), seed=42, resolution=1.0, fdr=config.fdr)
        display(pd.read_csv(resolve_path(SCZ_LEIDEN/"clusters.tsv"), sep="\t"))
        print("Excluded singletons:", ", ".join(pd.read_csv(resolve_path(SCZ_LEIDEN/"excluded_singletons.tsv"), sep="\t").symbol.astype(str)))
        from IPython.display import Image
        display(Image(filename=str(resolve_path(SCZ_LEIDEN/"leiden_clusters.png"))))
        print("Annotated Leiden graph, memberships and enrichment:", SCZ_LEIDEN)
    else:
        print("Leiden shared-core analysis requires both SCHEMA and GWAS targets.")


### Pathway, tissue, and cell-type context

Annotate pair networks and the shared core with frozen GO, Reactome, SynGO,
MitoCarta, and HPA sets. Hypergeometric ORA uses the full tested network, with
BH correction within annotation group across all queried networks.
`gene_context.tsv` records HPA specificity and network membership.


In [ ]:
if os.environ.get("CNV_RUN_LEGACY_SHARED_CORE") == "1":
    from cnv_overlap.results import resolve_path
    SCZ_CONTEXT = None
    if "SCZ_NETWORKS" in globals() and both_targets:
        from shared_network_cluster_context import annotate_network_context
        SCZ_CONTEXT = annotate_network_context(SCZ_NETWORKS, annotation_dir=annotation_dir,
            output_root=results_dir(RESULTS_ROOT, "scz_network_cluster_context"), fdr=config.fdr)
        with pd.option_context("display.max_colwidth", 200, "display.max_rows", None):
            display(pd.read_csv(resolve_path(SCZ_CONTEXT/"network_context.tsv"), sep="\t").filter(regex=r"^(network|n_genes|top_)"))
        display(pd.read_csv(resolve_path(SCZ_CONTEXT/"gene_context.tsv"), sep="\t"))
        from IPython.display import Image
        display(Image(filename=str(resolve_path(SCZ_CONTEXT/"tissue_celltype_enrichment.png"))))
        print("Colocalised-network context tables and tissue/cell-type heatmap:", SCZ_CONTEXT)
    else:
        print("Network context requires the SCZ colocalised networks with both SCHEMA and GWAS targets.")


## Shared-core threshold sensitivity

Test cores supported by ≥2, ≥3, or ≥4 CNVs with SCHEMA, GWAS, either, or both
targets. Each degree-matched draw rebuilds all definitions with fixed SCZ
profiles. Core-size empirical p-values are BH-adjusted across the 12 variants.
The primary definition requires ≥3 CNVs with both targets at the same gene.


In [ ]:
if os.environ.get("CNV_RUN_LEGACY_SHARED_CORE") == "1":
    from cnv_overlap.results import resolve_path
    if SCZ_SENSITIVITY is not None:
        from IPython.display import Image
        core_tests = pd.read_csv(resolve_path(SCZ_SENSITIVITY/"core_size_tests.tsv"), sep="\t")
        display(core_tests.drop(columns="genes"))
        display(Image(filename=str(resolve_path(SCZ_SENSITIVITY/"core_sensitivity.png"))))
        if (resolve_path(SCZ_SENSITIVITY/"enrichment_significant.tsv")).exists():
            sensitivity_terms = pd.read_csv(resolve_path(SCZ_SENSITIVITY/"enrichment_significant.tsv"), sep="\t")
            display(sensitivity_terms.sort_values("q_all_variants_libraries").head(40))
        print("Sensitivity memberships, null draws and enrichment:", SCZ_SENSITIVITY)
    else:
        print("Core sensitivity requires both SCHEMA and GWAS targets.")


### Shared-network pathways, biological processes and molecular functions

In [ ]:
if os.environ.get("CNV_RUN_LEGACY_SHARED_CORE") == "1":
    from cnv_overlap.results import resolve_path
    if SCZ_ENRICHMENT is not None:
        from IPython.display import Image
        enrichment = pd.read_csv(resolve_path(SCZ_ENRICHMENT/"enrichment_all_terms.tsv"), sep="\t")
        display(pd.read_csv(resolve_path(SCZ_ENRICHMENT/"annotation_coverage.tsv"), sep="\t"))
        for library in LIBRARIES:
            print(library)
            display(enrichment[(enrichment.library == library) & (enrichment.q_all_libraries < config.fdr)]
                    [["term", "overlap", "query_size", "fold_enrichment", "q_all_libraries", "genes"]])
        display(Image(filename=str(resolve_path(SCZ_ENRICHMENT/"enrichment_summary.png"))))
        print("All enrichment results:", SCZ_ENRICHMENT)
    else:
        print("Shared-network enrichment was not run; see target/library availability above.")


## Leave-one-CNV-out stability

Recompute observed convergence after omitting each CNV to identify influential
loci. These are descriptive sensitivity summaries.


In [ ]:
from cnv_overlap.results import resolve_path
leave_one_out = pd.read_csv(resolve_path(CONVERGENCE / "leave_one_out.tsv"), sep="\t")
display(leave_one_out)
# A descriptive community network built from genes reached by at least two CNVs.
from cnv_overlap.interpretation import candidate_network
multi_candidates = convergence_genes.loc[convergence_genes.n_cnvs >= 2, ["entrez"]]
MULTI_G, multi_membership = candidate_network(A, nodes, multi_candidates)
multi_membership = multi_membership.merge(convergence_genes, on="entrez", how="left")
multi_membership.to_csv(resolve_path(FIGURES / "multi_cnv_communities.tsv"), sep="\t", index=False)
display(multi_membership.head(40))


## Optional contiguous-gene-window sensitivity (off by default)

In [ ]:
from cnv_overlap.results import resolve_path
RUN_REGION_SENSITIVITY = False
if RUN_REGION_SENSITIVITY:
    REGION_REPS = min(1000, config.permutation_reps)
    REGION_CONVERGENCE = run_convergence(inputs, config=config, cache_dir=root/spec["cache_dir"],
        output_root=results_dir(RESULTS_ROOT, "multi_cnv_convergence")/"region", reps=REGION_REPS,
        null_model="region", scz_entities=scz_entities)
    region_tests = pd.read_csv(resolve_path(REGION_CONVERGENCE / "convergence_tests.tsv"), sep="\t")
    display(pd.concat([convergence_tests.assign(null_model="degree"),
                       region_tests.assign(null_model="contiguous_gene_window")], ignore_index=True))
    display(pd.read_csv(resolve_path(REGION_CONVERGENCE / "seed_qc.tsv"), sep="\t"))
else:
    print("Optional contiguous-gene-window sensitivity not run.")


## Supporting CNV–CNV pair tests

Colour shows the COLOC score (observed / mean permuted network size) on a log2
scale. Stars mark size-test BH significance within this comparison family.
`p` and `q` use the NetColoc normal tail; empirical tails and their intervals
are permutation diagnostics. Constant nulls have missing p/q.


In [ ]:
from cnv_overlap.results import resolve_path
tests = pd.read_csv(resolve_path(RUN / "pair_tests.tsv"), sep="\t")
PRIMARY_NULL = "netcoloc"
primary = tests[(tests["null"] == PRIMARY_NULL) & (tests.statistic == "size")].copy()
for family in ["region_pair"]:
    if (primary.family == family).any():
        fig = pair_heatmap(RUN, null=PRIMARY_NULL, family=family, fdr=config.fdr)
        fig.savefig(resolve_path(FIGURES / f"{family}.png"), bbox_inches="tight")
        fig.savefig(resolve_path(FIGURES / f"{family}.pdf"), bbox_inches="tight")
        plt.show()
        fig = pair_heatmap(RUN, null=PRIMARY_NULL, family=family, fdr=config.fdr, metric="ratio_genes")
        fig.savefig(resolve_path(FIGURES / f"{family}_genes.png"), bbox_inches="tight")
        fig.savefig(resolve_path(FIGURES / f"{family}_genes.pdf"), bbox_inches="tight")
        plt.show()
display(primary.assign(size_ratio=primary.observed / primary.null_mean).sort_values(["family", "q"])[["a", "b", "family", "observed", "null_mean", "size_ratio", "excess", "p", "q", "p_empirical", "tail_ci_low", "tail_ci_high", "status"]])


## Profile summaries and candidate genes

Signed similarity includes shared negative scores. Candidate genes meet the
configured joint thresholds; their recurrence is descriptive.


In [ ]:
from cnv_overlap.results import resolve_path
display(tests[tests.statistic != "size"].sort_values(["null", "statistic", "q"]).head(30))
candidates = pd.read_csv(resolve_path(RUN / "candidate_genes.tsv"), sep="\t")
selected = set(zip(primary.loc[primary.q < config.fdr, "a"], primary.loc[primary.q < config.fdr, "b"]))
selected_candidates = candidates.loc[np.array([pair in selected for pair in zip(candidates.a, candidates.b)], dtype=bool)]
display(selected_candidates.head(40))
recurrence = pd.read_csv(resolve_path(RUN / "recurrence.tsv"), sep="\t")
display(recurrence.sort_values(["n_regions", "n_pairs"], ascending=False).head(30))


## Descriptive communities

Partition each connected component separately and summarize candidate-gene
recurrence, degree, and community membership.


In [ ]:
from cnv_overlap.results import resolve_path
from cnv_overlap.interpretation import candidate_network
G, membership = candidate_network(A, nodes, selected_candidates)
membership["symbol"] = membership.entrez.map(symbols)
membership.to_csv(resolve_path(FIGURES / "candidate_communities.tsv"), sep="\t", index=False)
print(f"{G.number_of_nodes()} candidate nodes, {G.number_of_edges()} edges")
display(membership.sort_values(["community", "degree_in_candidate_network"], ascending=[True, False]).head(40))


## Offline annotation

Annotate with explicit GMT snapshots using `cnv-overlap annotate`. Descriptive
ORA uses the tested network universe as its background.


In [ ]:
from cnv_overlap.results import resolve_path
from notebook_analysis import export_versioned_report
GMT_PATHS = [root / "input/annotations" / name for name in
             ["GO_Biological_Process_2026.gmt", "ReactomePathways.gmt"]]
if not all(path.exists() for path in GMT_PATHS):
    print("Annotation snapshots missing; exporting figures and networks only.")
    GMT_PATHS = []
REPORT = export_versioned_report(RUN, inputs, output_dir=results_dir(RESULTS_ROOT, "cnv_cnv_report"),
                       gmt_paths=GMT_PATHS, fdr=config.fdr)
print("Heatmaps, pair networks and annotation outputs:", REPORT)
if GMT_PATHS:
    annotation = pd.read_csv(resolve_path(REPORT / "enrichment/descriptive_ora.tsv"), sep="\t")
    selected_pairs = primary.loc[primary.q < config.fdr, ["a", "b"]]
    displayed = annotation.merge(selected_pairs, on=["a", "b"])
    display(displayed[displayed.q_all_pair_terms < config.fdr].sort_values("q_all_pair_terms").head(40))


In [ ]:
from cnv_overlap.results import finalize_manifest
from cnv_overlap.provenance import file_hash
figure_sources = globals().get("RUNS", {"main": globals().get("RUN")})
finalize_manifest(FIGURES / "manifest.json", {"status": "complete",
    "identity": {"network": NETWORK, "notebook": "cnv_convergence.ipynb", "notebook_sha256": file_hash(ROOT / "notebooks" / "cnv_convergence.ipynb")},
    "sources": {name: {"path": str(source.resolve()), "manifest_sha256": file_hash(source / "manifest.json")}
                for name, source in figure_sources.items() if source is not None}})
